# 04 — Features históricas e análise de recorrência
> **LAB-VET-01 — estudo com PaySim (dados sintéticos).** Não use dados confidenciais, credenciais ou identificadores reais neste repositório. Execute as células em ordem. Os resultados históricos estão documentados separadamente; as saídas foram removidas para publicação.

**Cuidado temporal:** não incluir operações do mesmo step ou de steps futuros; os rótulos são usados apenas na análise retrospectiva.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Coloque o CSV público PaySim em data/ (não versionar arquivos de dados).
DATA_PATH = Path("data/PS_20174392719_1491204439457_log.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset ausente: {DATA_PATH.resolve()}. Consulte o README.")
df = pd.read_csv(DATA_PATH)
print("Dataset:", df.shape)

In [ ]:
treino = df.loc[df["step"] <= 300].copy()
validacao = df.loc[(df["step"] > 300) & (df["step"] <= 400)].copy()
teste = df.loc[df["step"] > 400].copy()
print("Treino:", treino.shape, "Validação:", validacao.shape, "Teste:", teste.shape)

## Exploração das primeiras 100 mil transações

In [ ]:
df_grafo = treino.loc[treino["step"] <= 100, ["step", "type", "amount", "nameOrig", "nameDest", "isFraud"]].copy()
df_grafo["tx_id"] = "TX_" + df_grafo.index.astype(str)
historico = df_grafo.head(100_000).copy()

In [ ]:
def adicionar_features_historicas(dados):
    """Usa somente transações de steps estritamente anteriores; não utiliza rótulos."""
    base = dados.copy()
    agregado = (base.groupby(["nameDest", "step"], sort=True)
                .agg(qtd_step=("amount", "size"), volume_step=("amount", "sum"))
                .reset_index().sort_values(["nameDest", "step"]))
    grupo = agregado.groupby("nameDest", sort=False)
    agregado["destino_tx_anteriores"] = grupo["qtd_step"].cumsum() - agregado["qtd_step"]
    agregado["destino_volume_anterior"] = grupo["volume_step"].cumsum() - agregado["volume_step"]
    base = base.merge(agregado[["nameDest", "step", "destino_tx_anteriores", "destino_volume_anterior"]], on=["nameDest", "step"], how="left", validate="many_to_one", sort=False)
    base["destino_sem_historico"] = (base["destino_tx_anteriores"] == 0).astype("int8")
    assert len(base) == len(dados)
    assert base[["destino_tx_anteriores", "destino_volume_anterior", "destino_sem_historico"]].notna().all().all()
    return base

In [ ]:
historico = adicionar_features_historicas(historico)
print(historico[["destino_tx_anteriores", "destino_volume_anterior", "destino_sem_historico"]].describe())

In [ ]:
comparacao_historica = (
    historico[
        historico["type"].isin(["TRANSFER", "CASH_OUT"])
    ]
    .groupby(["type", "isFraud"])
    .agg(
        quantidade=("tx_id", "count"),
        media_historica=(
            "destino_tx_anteriores", "mean"
        ),
        mediana_historica=(
            "destino_tx_anteriores", "median"
        ),
        max_historico=(
            "destino_tx_anteriores", "max"
        )
    )
    .round(2)
)

print(comparacao_historica)

In [ ]:
analise = historico[
    historico["type"].isin(["TRANSFER", "CASH_OUT"])
].copy()

analise["destino_sem_historico"] = (
    analise["destino_tx_anteriores"] == 0
)

resultado = (
    analise.groupby(
        ["type", "destino_sem_historico"]
    )
    .agg(
        transacoes=("isFraud", "size"),
        fraudes=("isFraud", "sum"),
        taxa_fraude=("isFraud", "mean")
    )
    .reset_index()
)

resultado["taxa_fraude_pct"] = (
    resultado["taxa_fraude"] * 100
).round(4)

print(
    resultado[
        [
            "type",
            "destino_sem_historico",
            "transacoes",
            "fraudes",
            "taxa_fraude_pct"
        ]
    ].to_string(index=False)
)

## Features para treino e validação — sem usar rótulos futuros

In [ ]:
base_features = pd.concat([treino.assign(_particao="treino"), validacao.assign(_particao="validacao")], ignore_index=True)
base_features = adicionar_features_historicas(base_features)
treino_grafo = base_features.loc[base_features["_particao"] == "treino"].drop(columns="_particao")
validacao_grafo = base_features.loc[base_features["_particao"] == "validacao"].drop(columns="_particao")
print(treino_grafo.shape, validacao_grafo.shape)

---
**Autoria**

```text
Alexandre Lotti
<!LOTTI>
```
